In [1]:
# pip install Flask


In [3]:
from flask import Flask, jsonify
from sqlalchemy import create_engine, func
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.orm import sessionmaker
import json
from datetime import datetime, timedelta

In [5]:
# Create an app instance
app = Flask(__name__)

# Set up the database connection
engine = create_engine('sqlite:///hawaii.sqlite')

# Reflect the tables
Base = automap_base()
Base.prepare(engine, reflect=True)

C:\Users\owner\AppData\Local\Temp\ipykernel_21000\3434388979.py:9: SADeprecationWarning: The AutomapBase.prepare.reflect parameter is deprecated and will be removed in a future release.  Reflection is enabled when AutomapBase.prepare.autoload_with is passed.
  Base.prepare(engine, reflect=True)


In [7]:
# Save references to the tables
Station = Base.classes.station
Measurement = Base.classes.measurement

# Create a session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

AttributeError: station

In [9]:

# Homepage route
@app.route('/')
def home():
    return (
        f"Welcome to the Climate API!<br>"
        f"Available Routes:<br>"
        f"/api/v1.0/precipitation<br>"
        f"/api/v1.0/stations<br>"
        f"/api/v1.0/tobs<br>"
        f"/api/v1.0/<start><br>"
        f"/api/v1.0/<start>/<end><br>"
    )

In [11]:

# Route for precipitation data
@app.route('/api/v1.0/precipitation')
def precipitation():
    # Query for the last 12 months of precipitation data
    recent_date = session.query(func.max(Measurement.date)).scalar()
    year_ago = datetime.strptime(recent_date, '%Y-%m-%d') - timedelta(days=365)
    precipitation_data = session.query(Measurement.date, Measurement.prcp).filter(Measurement.date >= year_ago).all()


In [13]:

    # Convert the results into a dictionary
    precip_dict = {date: prcp for date, prcp in precipitation_data}

    return jsonify(precip_dict)

NameError: name 'precipitation_data' is not defined

In [15]:
@app.route('/api/v1.0/stations')
def stations():
    # Query for all stations
    stations = session.query(Station.station).all()
    
    # Return a JSON list of stations
    return jsonify([station[0] for station in stations])


In [17]:
@app.route('/api/v1.0/tobs')
def tobs():
    # Find the most active station
    active_station = session.query(Measurement.station, func.count(Measurement.station))\
        .group_by(Measurement.station).order_by(func.count(Measurement.station).desc()).first()
    
    most_active_station_id = active_station[0]
    
    # Get the temperature observations for the most active station in the last 12 months
    recent_date = session.query(func.max(Measurement.date)).scalar()
    year_ago = datetime.strptime(recent_date, '%Y-%m-%d') - timedelta(days=365)
    temps = session.query(Measurement.date, Measurement.tobs)\
        .filter(Measurement.station == most_active_station_id)\
        .filter(Measurement.date >= year_ago).all()
    
    # Return the temperature observations in JSON format
    return jsonify([{"date": date, "tobs": tobs} for date, tobs in temps])


In [19]:
@app.route('/api/v1.0/<start>')
def start(start):
    # Query the minimum, average, and maximum temperatures for the given start date
    results = session.query(
        func.min(Measurement.tobs),
        func.avg(Measurement.tobs),
        func.max(Measurement.tobs)
    ).filter(Measurement.date >= start).all()
    
    # Return the results in JSON format
    return jsonify({
        "TMIN": results[0][0],
        "TAVG": results[0][1],
        "TMAX": results[0][2]
    })


In [21]:
if __name__ == '__main__':
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\owner\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
